# Apache Iceberg - Python

All 10 Python examples from [docs/core/iceberg.md](https://platob.github.io/rkp/core/iceberg/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table, assign_field_ids

# Iceberg resolves a column by identifier, so a schema is numbered first.
schema = assign_field_ids(pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("venue", pa.string()),
]))

root = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades")

# A table is created in a folder, and a folder is all it ever touches.
table = Table.create(root, schema, ["venue"])

# A table that has never been written to has no current snapshot.
assert table.current_snapshot is None
assert table.scan().read_all().num_rows == 0

table.append(
    pa.record_batch(
        {"id": [1, 2], "venue": ["XNAS", "XNYS"]},
        schema=pa.schema([
            pa.field("id", pa.int64(), nullable=False),
            pa.field("venue", pa.string()),
        ]),
    )
)

assert table.current_snapshot is not None
assert table.current_snapshot.operation == "append"
assert len(table.data_files()) == 2, "one file per venue"

# Reopening finds the table again, with no catalog in between.
reopened = Table.open(IOBase(root.url.to_path()))
assert reopened.scan().read_all().num_rows == 2

## What a table writes

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table, assign_field_ids

schema = assign_field_ids(pa.schema([pa.field("id", pa.int64(), nullable=False)]))
root = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades")

table = Table.create(root, schema)
table.append(
    pa.record_batch(
        {"id": [1]}, schema=pa.schema([pa.field("id", pa.int64(), nullable=False)])
    )
)

# `table.root` is the folder handle the table reads and writes through.
names = [
    entry.name
    for entry in table.root.ls(recursive=True)
    if entry.is_file()
]

# One Parquet data file, one manifest, one manifest list, two metadata
# documents (create, then commit), and the version hint that finds them.
assert any(name.endswith(".parquet") for name in names)
assert any(name.startswith("snap-") and name.endswith(".avro") for name in names)
assert any(name.endswith("-m0.avro") for name in names)
assert "v1.metadata.json" in names
assert "v2.metadata.json" in names
assert "version-hint.text" in names

## Manifest lists and manifests

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table, assign_field_ids

columns = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("venue", pa.string()),
])
schema = assign_field_ids(columns)

root = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades")
table = Table.create(root, schema, ["venue"])
table.append(
    pa.record_batch({"id": [1, 2], "venue": ["XNAS", "XNAS"]}, schema=columns)
)

# A snapshot names one manifest list; each of its rows is a manifest.
manifests = table.manifests()
assert len(manifests) == 1
assert manifests[0].is_data()
assert manifests[0].added_files_count == 1
assert manifests[0].added_rows_count == 2

# Each manifest row is a data file plus what the writer measured about it.
(file, spec), = table.data_files()
assert file.file_format == "PARQUET"
assert file.record_count == 2
assert spec.fields[0].name == "venue"

# Statistics are keyed by field id, which is what lets a planner skip a file.
assert file.value_counts[1] == 2
assert 1 in file.column_sizes

## Partition specs and the Hive layout

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table, assign_field_ids

columns = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("venue", pa.string()),
])
schema = assign_field_ids(columns)

root = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades")
table = Table.create(root, schema, ["venue"])
table.append(pa.record_batch({"id": [1, 2], "venue": ["XNAS", None]}, schema=columns))

files = table.data_files()
assert len(files) == 2
null_file, _ = next(pair for pair in files if pair[0].partition[0] is None)
assert "venue=null" in null_file.path, "the path spells it"
assert null_file.partition[0] is None, "the manifest means it"

## Reading with column pushdown

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table, assign_field_ids

columns = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("symbol", pa.string()),
])
schema = assign_field_ids(columns)

root = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades")
table = Table.create(root, schema)
table.append(
    pa.record_batch({"id": [1, 2], "symbol": ["AAPL", "MSFT"]}, schema=columns)
)

# The target names the columns to keep; each file's Parquet reader gets it as
# its own projection mask, so the dropped column chunk is never decoded.
wanted = pa.schema([pa.field("id", pa.int64(), nullable=False)])
reader = table.scan(wanted)
assert reader.schema.names == ["id"]
assert reader.read_all().num_rows == 2

# No target reads everything.
assert table.scan().schema.names == ["id", "symbol"]

## The three record methods over a table

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table, assign_field_ids

columns = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("venue", pa.string()),
])
schema = assign_field_ids(columns)
rows = lambda ids, venues: pa.record_batch(
    {"id": ids, "venue": venues}, schema=columns
)

path = pathlib.Path(tempfile.mkdtemp()) / "trades"
Table.create(IOBase(path), schema, ["venue"])

# The folder *is* the table, so the ordinary record surface reaches it. Its
# options come from the metadata, before a single data file exists.
folder = IOBase(path)
options = folder.record_options()
folder.write_arrow_batch_reader(rows([1, 2], ["XNAS", "XNYS"]), options=options)
folder.append_arrow_batch_reader(rows([3], ["XLON"]), options=options)

# A match key upserts: `2` is stored and updates, `9` is new and appends.
merging = folder.record_options()
merging.merge_by = ["id"]
folder.write_arrow_batch_reader(rows([2, 9], ["XNYS", "XLON"]), options=merging)

assert folder.read_arrow_batch_reader(options=options).read_all().num_rows == 4

# Each call was one commit, and the read went through the last one.
assert len(Table.open(IOBase(path)).snapshots) == 3

## Schema evolution and field ids

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table, assign_field_ids

columns = pa.schema([pa.field("id", pa.int64(), nullable=False)])
schema = assign_field_ids(columns)

root = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades")
table = Table.create(root, schema)
table.append(pa.record_batch({"id": [1]}, schema=columns))

# Add a column. Numbering continues above `last-column-id`, so the new column
# can never be confused with a dropped one.
evolved = assign_field_ids(pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("quantity", pa.int64()),
]))
assert table.evolve_schema(evolved) == 1, "the new schema's id"

# The old schema is retained, so the snapshot written under it still reads.
assert len(table.schemas) == 2
assert len(table.schemas[0].data_type) == 1

# And the file written before the column existed reads it as null.
rows = table.scan().read_all()
assert rows.column_names == ["id", "quantity"]
assert rows.column("quantity").null_count == rows.num_rows

In [ ]:
import pyarrow as pa

from yggdryl.iceberg import assign_field_ids

columns = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field(
        "leg",
        pa.struct([pa.field("price", pa.decimal128(18, 4), nullable=False)]),
    ),
])

# Depth first from `start`; the numbered schema is what comes back, so the
# schema handed in is left as it was.
schema = assign_field_ids(columns, 1)
assert [child.id for child in schema.data_type] == [1, 2]
assert schema.data_type[1].data_type[0].id == 3

# The root is not a column, so it is not numbered.
assert schema.id is None

# A field that already carries an id keeps it, so a second pass changes nothing.
assert [child.id for child in assign_field_ids(schema, 100).data_type] == [1, 2]

In [ ]:
import pathlib
import tempfile

import pyarrow as pa
import pytest

from yggdryl import IOBase
from yggdryl.iceberg import Table

columns = pa.schema([pa.field("id", pa.int64(), nullable=False)])

with pytest.raises(ValueError, match="assign_field_ids"):
    Table.create(IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades"), columns)

## Schemas as documents

In [ ]:
import json

from yggdryl.iceberg import schema_from_json, schema_to_json

document = json.loads("""{"type":"struct","schema-id":0,"fields":[
    {"id":1,"name":"id","required":true,"type":"long"},
    {"id":2,"name":"symbol","required":false,"type":"string"}
]}""")

# An Iceberg schema is a non-null struct field; its columns are the children.
schema = schema_from_json("row", document)
assert schema.data_type.kind == "struct"
assert not schema.nullable
assert len(schema.data_type) == 2
assert str(schema.data_type[0].data_type) == "int64"

# `required` inverts into nullability, and `id` becomes PARQUET:field_id.
assert not schema.data_type[0].nullable
assert schema.data_type[1].nullable
assert schema.data_type[0].id == 1
assert schema.data_type[0]["PARQUET:field_id"] == "1"

# The same document comes back out.
assert schema_to_json(schema) == document